In [1]:
#required imports
from sklearn.metrics import roc_auc_score
import glob, os, math, sys,json,time,gc
import numpy as np, pandas as pd
from pathlib import Path
import torch
import tqdm
import time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#safety checks to avoid memory bottlenecks
# torch.backends.cudnn.benchmark = False      # disable the exhaustive search
# torch.backends.cudnn.deterministic = True   # pick a deterministic kernel
# !export CUDNN_WORKSPACE_LIMIT_IN_MB=512   # or 256
# !export CUDNN_BENCHMARK=0

In [2]:
#declare gene-drug map
single_drugs = {
    "rifampicin" : ["rpoB"],
    "pyrazinamide": ["pncA"],
    "capreomycin" : ["tlyA"],
    "amikacin"    : ["eis"]
}

multi_drugs = {
    "streptomycin": ["rpsL", "gid"],
    "isoniazid"   : ["katG", "inhA"],
    "ethionamide" : ["ethA", "ethR","inhA"],
    "ethambutol"  : ["embC","embA","embB"],
    "moxifloxacin": ["gyrA", "gyrB"],
    "levofloxacin": ["gyrA", "gyrB"]
}

all_drugs = {**single_drugs, **multi_drugs}   # merge dicts

## Step 1: convert to float16 memory mapping and meta files for efficient memory

In [7]:
# ──────────────────────────────────────────────────────────────
# 0.  one‑time conversion: NPZ  →  float16 mem‑map + meta
# ──────────────────────────────────────────────────────────────
#     *.npz files that hold token tensors  →  lightweight
#     float-16 memory-mapped arrays + a tiny “meta” file.
#
#     Why ?
#       • fp16 cuts RAM / disk roughly in half.
#       • mmap lets PyTorch stream slices straight off disk.
# ──────────────────────────────────────────────────────────────


# -------------------------------------------------------------------
# 1) only the exceptions live here
#    (key = (gene, drug)  →  sub-folder to prepend)
# -------------------------------------------------------------------
SPECIAL_DIRS = {
    ("gyrA", "levofloxacin") : "data/latest/embeddings/gyrA_LEV",
    ("gyrB", "levofloxacin") : "data/latest/embeddings/gyrB_LEV",
    ("gyrA", "moxifloxacin") : "data/latest/embeddings/gyrA_MOX",
    ("gyrB", "moxifloxacin") : "data/latest/embeddings/gyrB_MOX",
    ("ethA", "ethionamide")  : "data/latest/embeddings/ethA_ETH",
    ("ethR", "ethionamide")  : "data/latest/embeddings/ethR_ETH",
    ("inhA", "ethionamide")  : "data/latest/embeddings/inhA_ETH",
    
}

# -------------------------------------------------------------------
# 2) one helper – return the directory that actually holds the .npz
# -------------------------------------------------------------------
def embeddings_root(gene: str, drug: str | None = None) -> Path:
    return Path(SPECIAL_DIRS.get((gene, drug),
                                 f"data/latest/embeddings/{gene}"))

# -------------------------------------------------------------------
# 3) the conversion itself – no branches inside the loop
# -------------------------------------------------------------------
def npz_to_memmap(gene: str, *, drug: str | None = None):
    root = embeddings_root(gene, drug)
    src_glob = root / "token" / f"{gene}_tok_chunk_*.npz"

    for npz in tqdm.tqdm(sorted(glob.glob(str(src_glob))), desc=f"convert {gene}"):
        meta_out = npz.replace(".npz", "_meta.npz")
        mmap_out = npz.replace(".npz", ".mmap")
        if os.path.exists(meta_out):        # skip if done
            continue

        z = np.load(npz)
        if "tokens" not in z:                # nothing to convert → skip
            print(f"  ⤷ skip {os.path.basename(npz)} (no 'tokens')")
            continue
        ids  = z["identifier"].astype(str)
        tok  = z["tokens"].astype("float16")      # cast to fp16 (½ RAM)
        mm   = np.memmap(mmap_out, mode="w+", dtype="float16",
                         shape=tok.shape)
        mm[:] = tok                             # write once
        mm.flush()
        np.savez_compressed(meta_out,
                            identifier=ids,
                            shape=tok.shape,
                            mmap_path=os.path.basename(mmap_out))
    print(f"{gene}: conversion finished")


In [ ]:
# # -------------------------------------------------------------------
# # Example calls
# # -------------------------------------------------------------------
# # “Normal” genes – use default directory
# # npz_to_memmap("katG")
# #
# # Special gyrA / gyrB cases
# # npz_to_memmap("gyrA", drug="moxifloxacin")
# # npz_to_memmap("gyrB", drug="levofloxacin")


special_drugs = {"levofloxacin", "moxifloxacin","ethionamide"}   # the ones with MOX / LEV folders

for drug, genes in all_drugs.items():
    print(f"\n===== {drug.upper()} =====")

    for g in genes:
        t0 = time.perf_counter()

        # -----------------------------------------------------------------
        # pass drug=drug only if this drug has the special folder structure
        # -----------------------------------------------------------------
        if drug in special_drugs:                 # safe, boolean expression
            npz_to_memmap(gene=g, drug=drug)      # keyword arguments ≠ positional
        else:
            npz_to_memmap(gene=g)                 # default drug=None inside function

        dt = time.perf_counter() - t0
        print(f"  {g}: conversion finished in {dt/60:.2f} min")


## Step 2: PCA 

In [ ]:
from sklearn.decomposition import IncrementalPCA
from tqdm import tqdm 

In [ ]:
# ──────────────────────────────────────────────────────────────
# joint PCA fit (only if .npz not yet present)
# 1)  Fit ONE PCA that sees all channels (320-D) from every gene
#     in the drug’s panel.  We save that basis once and re-use.
# ──────────────────────────────────────────────────────────────

def fit_joint_pca(genes, K,
                  drug: str | None = None,          # <── NEW
                  batch_res: int = 10_000,
                  overwrite: bool = False) -> Path:
    """
    genes        : ['katG', 'inhA']  (order doesn’t matter)
    K            : how many PCs we want (e.g. 10)
    batch_res    : how many (L,320) rows to stream per partial_fit
    root         : top-level folder that holds the token chunks
    overwrite    : set True if we want to re-fit even when a file exists

    Output: writes   <root>/<katG_inhA>_pc10.npz   with
                 • mean      (shape 320)
                 • components (shape K×320)
    """

    #---- filename that will hold the PCA weights-----
    tag        = "_".join(genes) # 'katg_inhA
    comp_path  = Path("data/latest/embeddings") / f"{tag}_pc{K}.npz"   # katG_inhA_pc10.npz
    
    # --- skip if we already have it -----
    if comp_path.exists() and not overwrite:
        print("PCA already fitted →", comp_path)
        return comp_path

    # ---------------
    # ipca = IncrementalPCA(...)
    # ---------------
    # IncrementalPCA lets us feed data in mini-batches instead of forming one big (ΣL x 320) matrix in RAM. 
    # It keeps running estimates of 
    #  • the global mean (for centering)
    #  • the top-k eigenvectors (principal components)
    # every call to partial_fit(batch) updates those estimates.

    # ipca = IncrementalPCA(n_components=K, batch_size=batch_res)
    ipca = IncrementalPCA(n_components=K, batch_size=None)

    # -----------------------------------------------------------------------
    # Walk through every *.mmap  file for every gene, stream rows, update PCA
    # -----------------------------------------------------------------------

    for g in genes:
        data_path=embeddings_root(g, drug)
        meta_paths = glob.glob(f"{data_path}/token/*_meta.npz")
        for mp in tqdm(meta_paths, desc=f"PCA fit {g}"):
            meta = np.load(mp, allow_pickle=True) # header info only
            mm   = np.memmap(mp.replace("_meta.npz", ".mmap"),
                             dtype="float16", mode="r", shape=tuple(meta["shape"])) # (N_isolates, L, 320)
            # loop over isolates inside this chunk
            for arr in mm:                         # arr: (L,320)
                # cast to float32 (IncrementalPCA needs FP32/64)
                ipca.partial_fit(arr.astype("float32"))
                #   ^ updates running mean & components using this batch
                #     NO full matrix ever lives in memory

    # Save the fitted basis → mean (320,)  and  components (K,320)
                
    np.savez(comp_path, 
             mean=ipca.mean_, # global channel-wise mean
             components=ipca.components_) # top-K eigenvectors
    print("joint PCA saved to", comp_path)
    return comp_path



In [ ]:
# ──────────────────────────────────────────────────────────────
# 2)  Take that PCA basis & project every chunk for ONE gene.
#     We stash the results under   .../token/PCA/
# ──────────────────────────────────────────────────────────────

def project_gene_to_pca(gene, comp_file, K, drug, root="data/latest/embeddings"):
    """
    gene       : e.g. 'gid'
    comp_file  : output of fit_joint_pca (…) – contains μ  &  W (K×320)
    K          : # components we kept
    root       : base folder that has   {gene}/token/*.mmap
    ----------------------------------------------------------------------------
    For every token chunk of <gene>:
        1) read the float16   (N_isolates , L , 320)   mmap
        2) subtract μ and right-multiply by Wᵀ  -->  (N , L , K)
        3) write a *new* float16 mmap into   {gene}/token/PCA/
           plus a tiny *_meta.npz   describing the file.
    ----------------------------------------------------------------------------
    Skips a chunk automatically if both output files already exist.
    """
    # ---------- 0) load PCA weights -----------------------------------------
    comp = np.load(comp_file)
    W, mu = comp["components"], comp["mean"]             # (K,320)  (320,)

    # ---------- 1) iterate over every *.mmap of this gene -------------------
    data_path = embeddings_root(g, drug) 
    meta_paths = sorted(Path(f"{data_path}/token").glob("*_meta.npz"))
    for mp in tqdm(meta_paths, desc=f"proj {gene}"):

        # ---------- target filenames inside ./PCA/ --------------------------
        pca_dir  = mp.parent / "PCA"             # …/gid/token/PCA/
        pca_dir.mkdir(exist_ok=True)

        base      = mp.stem.replace("_meta", "")              # gid_tok_chunk_0
        out_meta  = pca_dir / f"{base}_pc{K}_meta.npz"
        out_mmap  = pca_dir / f"{base}_pc{K}.mmap"

        # ---- skip if both files exist and sizes match ----------
        if out_meta.exists() and out_mmap.exists():
            continue
            
        # ---------- 2) open source mmap -------------------------------------
        meta  = np.load(mp, allow_pickle=True)
        # mm_in = np.memmap(mp.with_suffix(".mmap"), dtype="float16", mode="r",
                          # shape=tuple(meta["shape"])) # (N,L,320)
        mm_in = np.memmap(str(mp).replace("_meta.npz", ".mmap"), dtype="float16",mode="r", shape=tuple(meta["shape"]))
        
        # ---------- 3) allocate destination mmap ---------------------------
        mm_out = np.memmap(out_mmap, dtype="float16", mode="w+",
                           shape=(meta["shape"][0], # N isolates
                                  meta["shape"][1],  # L residues
                                  K)) # K PCs

        # ---------- 4) stream over isolates, project each ------------------
        for i in range(meta["shape"][0]): # loop rows
            X = mm_in[i].astype("float32") - mu # centre (L,320)
            mm_out[i] = (X @ W.T).astype("float16") # → (L,K) stored fp16

        mm_out.flush()  # ensure data is on disk
        
        # ---------- 5) tiny header so DataLoader knows the shape -----------
        np.savez(out_meta,
                 identifier=meta["identifier"], # same order as source
                 shape=mm_out.shape,
                 mmap_path=out_mmap.name) # just basename, not abs path




In [ ]:
# ------------------------------------------------------------
# 3) run for every drug (single & multi)
K = 10 #change k as we want
for drug, genes in all_drugs.items():
    # 1) fit one PCA on the concatenated (320-D) channel space
    #    using ALL sequences from all genes in *this* drug group
    comp_file = fit_joint_pca(genes, K, drug)  # works for single genes too  # returns path to .npz
    # 2) project every gene’s token mem-maps into that PCA basis
    for g in genes:
        project_gene_to_pca(g, comp_file, K,drug)
    print(f" {drug}: PCA done\n")


## Step 3: Mean across dimension (n,L,1) - Channel mean

In [ ]:
# ------------------------------------------------------------------
#   project 320-dim token chunks → 1-dim (mean) tokens
# ------------------------------------------------------------------
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

def project_gene_to_mean(gene, drug=None, root="data/latest/embeddings"):
    """
    Creates a <gene>_tok_chunk_*_mean.mmap next to every original chunk.
    Works for both the normal layout  data/embeddings/<gene>
    and the special gyrA/gyrB – levo/moxi folders via embeddings_root().
    """
    root_dir = embeddings_root(gene, drug)         
    meta_paths = sorted((root_dir / "token").glob("*_meta.npz"))
    if not meta_paths:
        raise FileNotFoundError(f"No *_meta.npz for {gene} in {root_dir}")

    for mp in tqdm(meta_paths, desc=f"mean {gene}"):
        mp        = Path(mp)
        meta_in   = np.load(mp, allow_pickle=True)
    
        # --- open input mmap correctly
        mmap_path = mp.parent / mp.name.replace("_meta.npz", ".mmap")
        mm_in     = np.memmap(mmap_path, dtype="float16", mode="r",
                              shape=tuple(meta_in["shape"]))        # (N,L,320)
    
        # --- prepare output files (skip if already done) ------------
        mean_dir  = mp.parent / "MEAN"
        mean_dir.mkdir(exist_ok=True)
        base      = mp.stem.replace("_meta", "")
        out_mmap = mean_dir / f"{base}_pcmean.mmap" 
        out_meta = mean_dir / f"{base}_pcmean_meta.npz" 
        if out_meta.exists() and out_mmap.exists():
            continue
    
        # --- write mean-compressed tensor ---------------------------
        mm_out = np.memmap(out_mmap, dtype="float16", mode="w+",
                           shape=(meta_in["shape"][0], meta_in["shape"][1], 1))
        mm_out[:] = mm_in.astype("float32").mean(axis=2, keepdims=True).astype("float16")
        mm_out.flush()
    
        np.savez(out_meta,
                 identifier = meta_in["identifier"],
                 shape      = mm_out.shape,
                 mmap_path  = out_mmap.name)
    
        del mm_out, mm_in      # close files



In [ ]:
for drug, genes in all_drugs.items():
    print(f"\n===== {drug.upper()} =====")

    for g in genes:
        t0 = time.perf_counter()

        # -----------------------------------------------------------------
        # pass drug=drug only if this drug has the special folder structure
        # -----------------------------------------------------------------
        if drug in special_drugs:                 # safe, boolean expression
            project_gene_to_mean(g, drug)      # keyword arguments ≠ positional
        else:
            project_gene_to_mean(g)                 # default drug=None inside function

        dt = time.perf_counter() - t0
        print(f"  {g}: conversion finished in {dt/60:.2f} min")